#  Movies exponential fits


## Step 0: Get the data (run this first)

This cell downloads IMDb's official non-commercial dataset directly from
[datasets.imdbws.com](https://datasets.imdbws.com/) (per IMDb's terms, for
personal/non-commercial use) and computes yearly movie counts per genre.
Everything is fetched fresh into your own Colab session — no IMDb-derived
data is hosted or redistributed by this notebook or site.

The download is about 200MB and may take a few minutes the first time.

In [ ]:
import os
import urllib.request
import pandas as pd

csv_path = "movies_by_genre_1980_2014.csv"

if not os.path.exists(csv_path):
    url = "https://datasets.imdbws.com/title.basics.tsv.gz"
    local_gz = "title.basics.tsv.gz"
    if not os.path.exists(local_gz):
        print("Downloading IMDb title.basics.tsv.gz (~200MB)... this may take a few minutes.")
        urllib.request.urlretrieve(url, local_gz)

    genres_of_interest = [
        "Romance", "Sport", "Horror", "Comedy", "Action", "Adventure",
        "Sci-Fi", "Crime", "Drama", "Family", "Mystery", "Thriller",
        "War", "Western", "Musical", "Documentary", "Biography", "Animation",
        "Fantasy", "History"
    ]

    yearly_counts = {genre: {} for genre in genres_of_interest}
    chunksize = 500000
    for chunk in pd.read_csv(local_gz, sep="\t", compression="gzip", chunksize=chunksize, dtype=str):
        movies = chunk[chunk["titleType"] == "movie"]
        movies = movies[movies["startYear"].str.isnumeric()].copy()
        movies["startYear"] = movies["startYear"].astype(int)
        movies = movies[(movies["startYear"] >= 1950) & (movies["startYear"] <= 2024)]

        for genre in genres_of_interest:
            mask = movies["genres"].str.contains(genre, na=False)
            counts = movies.loc[mask, "startYear"].value_counts().to_dict()
            for year, count in counts.items():
                yearly_counts[genre][year] = yearly_counts[genre].get(year, 0) + count

    counts_df = pd.DataFrame(yearly_counts).fillna(0).astype(int).sort_index()
    counts_df.index.name = "Year"
    counts_df = counts_df.reset_index()

    # Save the 1980-2014 slice used by the rest of this notebook
    counts_df[(counts_df["Year"] >= 1980) & (counts_df["Year"] <= 2014)].to_csv(csv_path, index=False)
    print(f"Saved {csv_path}")
else:
    print(f"Found existing {csv_path}, skipping download.")

In [ ]:
# --- Setup (Colab-friendly) ---
# If you're in Colab, Plotly and SciPy are already available.
# If your file isn't in the current directory, click the folder icon on the left
# and upload: movies_by_genre_1980_2014.csv

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import os

# Optional helper for Colab: uncomment to open a file picker
# from google.colab import files
# uploaded = files.upload()  # then select movies_by_genre_1980_2014.csv

# --- Load data ---
csv_path = "movies_by_genre_1980_2014.csv"  # make sure the name matches exactly
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Couldn't find 'movies_by_genre_1980_2014.csv'. "
        "Upload it to the current folder or adjust csv_path."
    )

df = pd.read_csv(csv_path)

# Basic sanity checks on expected columns
expected_cols = {"Year", "Romance"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}. "
                     "Check capitalization (e.g., 'Romance', not 'romance').")

# Keep only 1980–2014 (inclusive), sorted by Year
df = df[(df["Year"] >= 1980) & (df["Year"] <= 2014)].sort_values("Year").copy()

# --- Build variables ---
# Independent variable t in years with t=0 at 1980
t = (df["Year"] - 1980).to_numpy(dtype=float)  # shape (N,)
y = df["Romance"].to_numpy(dtype=float)        # shape (N,)

# --- Exponential model f(t) = a * b^t ---
def f_model(t, a, b):
    return a * (b ** t)

# Initial guesses:
# a0: value near the first data point; b0: rough growth factor per year
a0 = max(y[0], 1e-6)  # avoid zero
# Estimate b0 via ratio of last to first over total years; clamp to reasonable range
years_span = max(t[-1] - t[0], 1.0)
b0_raw = (max(y[-1], 1e-6) / max(y[0], 1e-6)) ** (1.0 / years_span)
b0 = float(np.clip(b0_raw, 0.5, 1.5))  # prevent wild starting guesses

# Fit with positivity constraints on a and b
popt, pcov = curve_fit(f_model, t, y, p0=[a0, b0], bounds=([0.0, 0.0], [np.inf, np.inf]))
a_hat, b_hat = popt

# --- Prepare smooth curve for plotting ---
t_line = np.linspace(t.min(), t.max(), 300)
y_fit = f_model(t_line, a_hat, b_hat)

# --- Print the fitted equation ---
# Round for readability but keep scientific notation if needed
def fmt(x):
    # compact formatting with up to 6 significant digits
    s = f"{x:.6g}"
    return s

print(f"Fitted model: f(t) = a * b^t  with  a = {fmt(a_hat)},  b = {fmt(b_hat)}")
print(f"So, f(t) = {fmt(a_hat)} * ({fmt(b_hat)})^t")

# (Optional) also report continuous form with e^(k t), where b = e^k
k_hat = np.log(b_hat)
print(f"Equivalent form: f(t) = {fmt(a_hat)} * e^({fmt(k_hat)} * t)")

# --- Interactive Plotly figure (t on the x-axis) ---
fig = go.Figure()

# Scatter of actual data
fig.add_trace(go.Scatter(
    x=t, y=y, mode="markers",
    name="Data (Romance count)",
    hovertemplate="t=%{x}<br>Romance=%{y}<extra></extra>"
))

# Smooth model curve
fig.add_trace(go.Scatter(
    x=t_line, y=y_fit, mode="lines",
    name="Exponential fit",
    hovertemplate="t=%{x:.2f}<br>f(t)=%{y:.2f}<extra></extra>"
))

fig.update_layout(
    title="Romance Movies per Year (1980–2014) with Exponential Fit",
    xaxis_title="t (years since 1980)",
    yaxis_title="Number of Romance Movies",
    legend=dict(
        x=0.01,  # Adjust x and y to position the legend
        y=0.99,
        bgcolor='rgba(255, 255, 255, 0.5)',  # Optional: add a semi-transparent background
        bordercolor='rgba(0, 0, 0, 0.5)',    # Optional: add a border
        borderwidth=1),
)

fig.show()


Fitted model: f(t) = a * b^t  with  a = 224.222,  b = 1.04736
So, f(t) = 224.222 * (1.04736)^t
Equivalent form: f(t) = 224.222 * e^(0.0462704 * t)


# all other genres

In [ ]:
# --- Setup (Colab-friendly) ---
# If you're in Colab, Plotly and SciPy are available.
# If your file isn't in the current directory, click the folder icon on the left
# and upload: movies_by_genre_1980_2014.csv

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import os

# Optional helper for Colab:
# from google.colab import files
# uploaded = files.upload()  # then select movies_by_genre_1980_2014.csv

# --- Load data ---
csv_path = "movies_by_genre_1980_2014.csv"  # make sure the name matches exactly
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Couldn't find 'movies_by_genre_1980_2014.csv'. "
        "Upload it to the current folder or adjust csv_path."
    )

df = pd.read_csv(csv_path)

# Sanity check: require 'Year' and 'Romance' columns (capitalization matters)
expected_cols = {"Year", "Romance"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}. "
                     "Check capitalization (e.g., 'Romance', not 'romance').")

# Keep only 1980–2014 and sort
df = df[(df["Year"] >= 1980) & (df["Year"] <= 2014)].sort_values("Year").copy()

# --- Build variables ---
# Independent variable t in years with t=0 at 1980
t = (df["Year"] - 1980).to_numpy(dtype=float)

# Identify genre columns (everything except 'Year')
genre_cols = [c for c in df.columns if c != "Year"]

# --- Exponential model and fitting helper ---
# f(t) = a * b^t
def f_model(t, a, b):
    return a * (b ** t)

def fit_exponential(t, y):
    # reasonable initial guesses
    a0 = max(float(y[0]), 1e-6)  # avoid zero
    span = max(float(t[-1] - t[0]), 1.0)
    b0_raw = (max(float(y[-1]), 1e-6) / max(float(y[0]), 1e-6)) ** (1.0 / span)
    b0 = float(np.clip(b0_raw, 0.5, 1.5))  # tame starting point
    # fit with positivity bounds
    (a_hat, b_hat), pcov = curve_fit(
        f_model, t, y, p0=[a0, b0],
        bounds=([0.0, 0.0], [np.inf, np.inf])
    )
    return a_hat, b_hat

def fmt(x):
    return f"{x:.6g}"  # compact up to 6 significant digits

# --- Fit Romance once (used on every plot) ---
y_rom = df["Romance"].to_numpy(dtype=float)
a_rom, b_rom = fit_exponential(t, y_rom)
k_rom = np.log(b_rom)
print("=== Fitted Equations ===")
print(f"Romance: f_rom(t) = {fmt(a_rom)} * ({fmt(b_rom)})^t   "
      f"(= {fmt(a_rom)} * e^({fmt(k_rom)} * t))")

# Smooth t for lines
t_line = np.linspace(t.min(), t.max(), 300)
y_rom_fit_line = f_model(t_line, a_rom, b_rom)

# --- For each OTHER genre, fit & plot with Romance overlaid ---
for genre in genre_cols:
    if genre == "Romance":
        continue

    y = df[genre].to_numpy(dtype=float)
    a_g, b_g = fit_exponential(t, y)
    k_g = np.log(b_g)

    print(f"{genre}: f_{genre}(t) = {fmt(a_g)} * ({fmt(b_g)})^t   "
          f"(= {fmt(a_g)} * e^({fmt(k_g)} * t))")

    # Model line for this genre
    y_fit_line = f_model(t_line, a_g, b_g)

    # --- Build interactive figure ---
    fig = go.Figure()

    # Data points
    # fig.add_trace(go.Scatter(
    #     x=t, y=y_rom, mode="markers", name="Romance data",
    #     hovertemplate="t=%{x}<br>Romance=%{y}<extra></extra>"
    # ))
    # fig.add_trace(go.Scatter(
    #     x=t_line, y=y_rom_fit_line, mode="lines", name="Romance fit",
    #     hovertemplate="t=%{x:.2f}<br>f_rom(t)=%{y:.2f}<extra></extra>"
    # ))
    fig.add_trace(go.Scatter(
        x=t, y=y, mode="markers", name=f"{genre} data",
        hovertemplate=f"t=%{{x}}<br>{genre}=%{{y}}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=t_line, y=y_fit_line, mode="lines", name=f"{genre} fit",
        hovertemplate=f"t=%{{x:.2f}}<br>f_{genre}(t)=%{{y:.2f}}<extra></extra>"
    ))

    fig.update_layout(
        title=f"{genre}  (1980–2014): Data and Exponential Fits",
        xaxis_title="t (years since 1980)",
        yaxis_title="Number of Movies",
        legend=dict(title=None),
    )

    fig.show()


=== Fitted Equations ===
Romance: f_rom(t) = 224.222 * (1.04736)^t   (= 224.222 * e^(0.0462704 * t))
Sport: f_Sport(t) = 5.05165 * (1.13284)^t   (= 5.05165 * e^(0.124731 * t))


Horror: f_Horror(t) = 38.2584 * (1.10678)^t   (= 38.2584 * e^(0.101452 * t))


Sci-Fi: f_Sci-Fi(t) = 22.3595 * (1.0822)^t   (= 22.3595 * e^(0.0789938 * t))


Family: f_Family(t) = 12.9134 * (1.12856)^t   (= 12.9134 * e^(0.120939 * t))


Mystery: f_Mystery(t) = 25.3546 * (1.09425)^t   (= 25.3546 * e^(0.0900701 * t))


Thriller: f_Thriller(t) = 83.018 * (1.08141)^t   (= 83.018 * e^(0.0782643 * t))


Documentary: f_Documentary(t) = 151.84 * (1.11584)^t   (= 151.84 * e^(0.109608 * t))


Biography: f_Biography(t) = 1.44032 * (1.22121)^t   (= 1.44032 * e^(0.199841 * t))


Animation: f_Animation(t) = 16.1338 * (1.08929)^t   (= 16.1338 * e^(0.0855227 * t))


History: f_History(t) = 1.73672 * (1.20375)^t   (= 1.73672 * e^(0.185442 * t))


## Growth rates by genre  (bar graph)



In [ ]:
# --- Setup (Colab-friendly) ---
# If you're in Colab, Plotly and SciPy are available.
# If your file isn't in the current directory, click the folder icon on the left
# and upload: movies_by_genre_1980_2014.csv

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import os

# Optional helper for Colab:
# from google.colab import files
# uploaded = files.upload()  # then select movies_by_genre_1980_2014.csv

# --- Load data ---
csv_path = "movies_by_genre_1980_2014.csv"  # make sure the name matches exactly
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Couldn't find 'movies_by_genre_1980_2014.csv'. "
        "Upload it to the current folder or adjust csv_path."
    )

df = pd.read_csv(csv_path)

# Sanity check: require 'Year' and 'Romance' columns (capitalization matters)
expected_cols = {"Year", "Romance"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}. "
                     "Check capitalization (e.g., 'Romance', not 'romance').")

# Keep only 1980–2014 and sort
df = df[(df["Year"] >= 1980) & (df["Year"] <= 2014)].sort_values("Year").copy()

# --- Build variables ---
# Independent variable t in years with t=0 at 1980
t = (df["Year"] - 1980).to_numpy(dtype=float)

# Identify genre columns (everything except 'Year')
genre_cols = [c for c in df.columns if c != "Year"]

# --- Exponential model and fitting helper ---
# f(t) = a * b^t
def f_model(t, a, b):
    return a * (b ** t)

def fit_exponential(t, y):
    # reasonable initial guesses
    a0 = max(float(y[0]), 1e-6)  # avoid zero
    span = max(float(t[-1] - t[0]), 1.0)
    b0_raw = (max(float(y[-1]), 1e-6) / max(float(y[0]), 1e-6)) ** (1.0 / span)
    b0 = float(np.clip(b0_raw, 0.5, 1.5))  # tame starting point
    # fit with positivity bounds
    (a_hat, b_hat), pcov = curve_fit(
        f_model, t, y, p0=[a0, b0],
        bounds=([0.0, 0.0], [np.inf, np.inf])
    )
    return a_hat, b_hat


# Collect growth rates (b values)
growth_rates = {}

# Calculate Romance growth rate
y_rom = df["Romance"].to_numpy(dtype=float)
a_rom, b_rom = fit_exponential(t, y_rom)

growth_rates['Romance'] = b_rom

for genre in genre_cols:
    if genre == "Romance":
        continue
    y = df[genre].to_numpy(dtype=float)
    a_g, b_g = fit_exponential(t, y)
    growth_rates[genre] = b_g

# Convert to pandas Series for easier plotting
growth_rates_series = pd.Series(growth_rates)

# Calculate r as a percentage
r_series = (growth_rates_series - 1) * 100

# Sort r values for better visualization
r_series = r_series.sort_values(ascending=True) # Sort ascending for horizontal bars

# Create bar graph using Plotly
fig = go.Figure(data=[go.Bar(
    x=r_series.values, # Use r values for the x-axis
    y=r_series.index,
    orientation='h', # Make bars horizontal
    text=[f'{val:.2f}%' for val in r_series.values], # Display percentage text
    textposition='outside' # Position text outside the bars
)])

fig.update_layout(
    title="Yearly Growth Rate (%) by Genre (1980-2014)",
    xaxis_title="Yearly Growth Rate (%) (r)", # Update axis title
    yaxis_title="Genre",
    yaxis={'categoryarray': r_series.index}, # Ensure correct order
    xaxis=dict(range=[0, 25]), # Set x-axis range from 0 to 25
    template='plotly_white' # Use a modern template with a white background
)

fig.show()

## plots of all genres with romance

In [ ]:
# --- Setup (Colab-friendly) ---
# If you're in Colab, Plotly and SciPy are available.
# If your file isn't in the current directory, click the folder icon on the left
# and upload: movies_by_genre_1980_2014.csv

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import os
import plotly.colors # Import plotly.colors

# Optional helper for Colab:
# from google.colab import files
# uploaded = files.upload()  # then select movies_by_genre_1980_2014.csv

# --- Load data ---
csv_path = "movies_by_genre_1980_2014.csv"  # make sure the name matches exactly
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Couldn't find 'movies_by_genre_1980_2014.csv'. "
        "Upload it to the current folder or adjust csv_path."
    )

df = pd.read_csv(csv_path)

# Sanity check: require 'Year' and 'Romance' columns (capitalization matters)
expected_cols = {"Year", "Romance"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}. "
                     "Check capitalization (e.g., 'Romance', not 'romance').")

# Keep only 1980–2014 and sort
df = df[(df["Year"] >= 1980) & (df["Year"] <= 2014)].sort_values("Year").copy()

# --- Build variables ---
# Independent variable t in years with t=0 at 1980
t = (df["Year"] - 1980).to_numpy(dtype=float)

# Identify genre columns (everything except 'Year')
genre_cols = [c for c in df.columns if c != "Year"]

# --- Exponential model and fitting helper ---
# f(t) = a * b^t
def f_model(t, a, b):
    return a * (b ** t)

def fit_exponential(t, y):
    # reasonable initial guesses
    a0 = max(float(y[0]), 1e-6)  # avoid zero
    span = max(float(t[-1] - t[0]), 1.0)
    b0_raw = (max(float(y[-1]), 1e-6) / max(float(y[0]), 1e-6)) ** (1.0 / span)
    b0 = float(np.clip(b0_raw, 0.5, 1.5))  # tame starting point
    # fit with positivity bounds
    (a_hat, b_hat), pcov = curve_fit(
        f_model, t, y, p0=[a0, b0],
        bounds=([0.0, 0.0], [np.inf, np.inf])
    )
    return a_hat, b_hat

def fmt(x):
    return f"{x:.6g}"  # compact up to 6 significant digits

# --- Fit Romance once (used on every plot) ---
y_rom = df["Romance"].to_numpy(dtype=float)
a_rom, b_rom = fit_exponential(t, y_rom)
k_rom = np.log(b_rom)
print("=== Fitted Equations ===")
print(f"Romance: f_rom(t) = {fmt(a_rom)} * ({fmt(b_rom)})^t   "
      f"(= {fmt(a_rom)} * e^({fmt(k_rom)} * t))")

# Smooth t for lines
t_line = np.linspace(t.min(), t.max(), 300)
y_rom_fit_line = f_model(t_line, a_rom, b_rom)

# --- Define colors for other genres ---
# Get a list of colors from Plotly's color scales (excluding blue)
# Using 'Plotly' scale and skipping the first color (which is blue)
colors = plotly.colors.qualitative.Plotly[1:]
# Ensure there are enough colors for all genres (excluding Romance)
if len(colors) < len(genre_cols) - 1:
    # If not enough colors, cycle through the available ones
    colors = colors * ((len(genre_cols) - 1) // len(colors) + 1)

# Create a dictionary to map genres to colors
genre_colors = {genre: color for genre, color in zip([g for g in genre_cols if g != 'Romance'], colors)}


# --- For each OTHER genre, fit & plot with Romance overlaid ---
for genre in genre_cols:
    if genre == "Romance":
        continue

    y = df[genre].to_numpy(dtype=float)
    a_g, b_g = fit_exponential(t, y)
    k_g = np.log(b_g)

    print(f"{genre}: f_{genre}(t) = {fmt(a_g)} * ({fmt(b_g)})^t   "
          f"(= {fmt(a_g)} * e^({fmt(k_g)} * t))")

    # Model line for this genre
    y_fit_line = f_model(t_line, a_g, b_g)

    # Get the color for the current genre
    current_color = genre_colors[genre]

    # --- Build interactive figure ---
    fig = go.Figure()

    # Data points
    fig.add_trace(go.Scatter(
        x=t, y=y_rom, mode="markers", name="Romance data",
        hovertemplate="t=%{x}<br>Romance=%{y}<extra></extra>",
        marker=dict(color='blue') # Set color to blue
    ))
    fig.add_trace(go.Scatter(
        x=t_line, y=y_rom_fit_line, mode="lines", name=f"Romance fit: {fmt(a_rom)} * ({fmt(b_rom)})^t",
        hovertemplate="t=%{x:.2f}<br>f_rom(t)=%{y:.2f}<extra></extra>",
        line=dict(color='blue') # Set color to blue
    ))
    fig.add_trace(go.Scatter(
        x=t, y=y, mode="markers", name=f"{genre} data",
        hovertemplate=f"t=%{{x}}<br>{genre}=%{{y}}<extra></extra>",
        marker=dict(color=current_color) # Set color to the assigned genre color
    ))
    fig.add_trace(go.Scatter(
        x=t_line, y=y_fit_line, mode="lines", name=f"{genre} fit: {fmt(a_g)} * ({fmt(b_g)})^t",
        hovertemplate=f"t=%{{x:.2f}}<br>f_{genre}(t)=%{{y:.2f}}<extra></extra>",
        line=dict(color=current_color) # Set color to the assigned genre color
    ))

    fig.update_layout(
        title=f"{genre} vs Romance (1980–2014): Data and Exponential Fits",
        xaxis_title="t (years since 1980)",
        yaxis_title="Number of Movies",
        legend=dict(
            x=0.01,  # Adjust x to position the legend
            y=0.99, # Adjust y to position the legend
            bgcolor='rgba(255, 255, 255, 0.5)',  # Optional: add a semi-transparent background
            bordercolor='rgba(0, 0, 0, 0.5)',    # Optional: add a border
            borderwidth=1),
    )

    fig.show()

=== Fitted Equations ===
Romance: f_rom(t) = 224.222 * (1.04736)^t   (= 224.222 * e^(0.0462704 * t))
Sport: f_Sport(t) = 5.05165 * (1.13284)^t   (= 5.05165 * e^(0.124731 * t))


Horror: f_Horror(t) = 38.2584 * (1.10678)^t   (= 38.2584 * e^(0.101452 * t))


Sci-Fi: f_Sci-Fi(t) = 22.3595 * (1.0822)^t   (= 22.3595 * e^(0.0789938 * t))


Family: f_Family(t) = 12.9134 * (1.12856)^t   (= 12.9134 * e^(0.120939 * t))


Mystery: f_Mystery(t) = 25.3546 * (1.09425)^t   (= 25.3546 * e^(0.0900701 * t))


Thriller: f_Thriller(t) = 83.018 * (1.08141)^t   (= 83.018 * e^(0.0782643 * t))


Documentary: f_Documentary(t) = 151.84 * (1.11584)^t   (= 151.84 * e^(0.109608 * t))


Biography: f_Biography(t) = 1.44032 * (1.22121)^t   (= 1.44032 * e^(0.199841 * t))


Animation: f_Animation(t) = 16.1338 * (1.08929)^t   (= 16.1338 * e^(0.0855227 * t))


History: f_History(t) = 1.73672 * (1.20375)^t   (= 1.73672 * e^(0.185442 * t))
